# Sizing and Matching a Compressor and Turbine with the Mass Flow Function

This notebook walks through `turbo-design`'s non-dimensional mass flow function end to end:

1. Size a compressor's inlet annulus directly from a target massflow and Mach number.
2. Solve the compressor with `turbo-design`'s radial-equilibrium solver.
3. Size a matching turbine's choked NGV throat, and match it to the compressor's shaft power and mass-flow continuity with `ShaftMatch`.
4. Sweep the compressor across a range of massflows to build a performance map, using `sweep_operating_points`.

See [`mass_flow_function.md`](https://github.com/nasa/turbo-design/blob/main/mass_flow_function.md) in the repo root for the full derivation and equations behind each step.

In [ ]:
!pip install -U turbo-design

## 1. Size the compressor inlet

The non-dimensional mass flow function

$$\tilde m(M,\gamma) = M\Big(1+\tfrac{\gamma-1}{2}M^2\Big)^{-\frac{\gamma+1}{2(\gamma-1)}}$$

inverts into an annulus area for a target massflow, total conditions, and Mach number:

$$A = \frac{\dot m \sqrt{T_0}}{P_0\sqrt{\gamma/R}\,\tilde m(M,\gamma)}$$

`area_for_massflow` computes this directly; `radii_for_area` then splits that area into hub/shroud radii around a chosen mean radius - the two pieces you need before a `Passage` even exists.

In [ ]:
import numpy as np

from turbodesign import Inlet, Outlet, Passage, PassageType, TurbineSpool
from turbodesign.compressor_spool import CompressorSpool
from turbodesign.flow_math import radii_for_area
from turbodesign.isentropic import area_for_massflow
from turbodesign.loss.fixedpressureloss import FixedPressureLoss
from turbodesign.row_factory import make_rotor_row, make_stator_row
from turbodesign.shaft_match import ComponentSizingSpec, ShaftMatch
from turbodesign.operating_map import sweep_operating_points

# Mattingly example 9.1: a single-stage axial compressor stage
T01_K = 518.7 / 1.8
P01_Pa = 14.7 * 6894.76
massflow_kg_s = 50 * 0.453592
M1 = 0.7
rmean = 12 * 0.0254

area1 = area_for_massflow(massflow_kg_s, P01_Pa, T01_K, M1, gamma=1.4, R=287.0)
rhub1, rshroud1 = radii_for_area(area1, rmean)
print(f"Station 1 (compressor inlet) area from MFP: {area1:.6f} m^2")
print(f"  r_hub = {rhub1:.4f} m, r_shroud = {rshroud1:.4f} m")

## 2. Build and solve the compressor

Stations 2 and 3 (rotor and stator exit) keep Mattingly's prescribed areas, since those come from
the stage's velocity-triangle/loading design, not a freely chosen Mach - MFP only sizes the inlet
boundary, where no blade row exists yet to hand you an area.

In [ ]:
area2 = 179.1 / 39.3701**2
area3 = 165.3 / 39.3701**2
h2 = area2 / (np.pi * 4 * rmean)
h3 = area3 / (np.pi * 4 * rmean)
cax = 1 * 0.0254

xhub_arr = [0, cax, 2 * cax]
xshroud_arr = [0, cax, 2 * cax]
rhub_arr = [rhub1, rmean - h2, rmean - h3]
rshroud_arr = [rshroud1, rmean + h2, rmean + h3]

passage = Passage(xhub_arr, rhub_arr, xshroud_arr, rshroud_arr, passageType=PassageType.Axial, zero_phi=True)
inlet = Inlet(hub_location=0)
inlet.alpha2 = [40]
inlet.init_total(P01_Pa, T01_K, M=M1)

rotor = make_rotor_row(hub_location=cax / max(xhub_arr), metal_exit_angle_deg=[-23.87], loss_function=FixedPressureLoss(0))
stator = make_stator_row(hub_location=2 * cax / max(xhub_arr), metal_exit_angle_deg=[40], loss_function=FixedPressureLoss(0))

outlet = Outlet()
outlet.init_total(1.3 * P01_Pa, 0.5)

rpm = 1000 * 30 / np.pi  # omega = 1000 rad/s
compressor = CompressorSpool(passage, massflow_kg_s, inlet, outlet, [rotor, stator], num_streamlines=1, rpm=rpm)
compressor.solve()

print(f"\nCompressor power: {compressor.total_power()/1000:.2f} kW")
print(f"Overall pressure ratio: {compressor.overall_pressure_ratio():.4f}")

## 3. Size and match a turbine

The nozzle guide vane (NGV) is the engine's metering orifice - by design, it runs choked (M=1), the
classic gas-generator sizing assumption. `ComponentSizingSpec` defaults `ngv_choked=True`, so the
station-4 area comes straight from the sonic value of $\tilde m$, with no target Mach to guess at.

`ShaftMatch` then scales a hand-built reference turbine's geometry (its angles, loss coefficients,
and stage layout are reused as-is - MFP gives an area, not a blade shape) and solves for the
(annulus area, exit pressure) pair that closes both the mass-flow and shaft-power residuals
together.

In [ ]:
def build_reference_turbine(rpm):
    """A hand-picked single-stage turbine whose annulus/exit-pressure ShaftMatch will
    scale to match the compressor - its angles and loss coefficients are reused verbatim."""
    T0, P0, P_exit, rmean = 1200.0, 126680.0, 90000.0, 0.15
    area1, area2, area3 = 0.05, 0.045, 0.05
    h1 = area1 / (np.pi * 4 * rmean)
    h2 = area2 / (np.pi * 4 * rmean)
    h3 = area3 / (np.pi * 4 * rmean)
    cax = 0.02

    xhub_arr = [0, cax, 2 * cax]
    xshroud_arr = [0, cax, 2 * cax]
    rhub_arr = [rmean - h1, rmean - h2, rmean - h3]
    rshroud_arr = [rmean + h1, rmean + h2, rmean + h3]
    passage = Passage(xhub_arr, rhub_arr, xshroud_arr, rshroud_arr, passageType=PassageType.Axial, zero_phi=True)

    inlet = Inlet(hub_location=0, alpha=[0])
    inlet.init_total(P0=P0, T0=T0, M=0.15)
    outlet = Outlet(num_streamlines=1)
    outlet.init_static(P=P_exit, percent_radii=[0.5])

    stator = make_stator_row(hub_location=cax / max(xhub_arr), metal_exit_angle_deg=[65.0], loss_function=FixedPressureLoss(0.03))
    rotor = make_rotor_row(hub_location=2 * cax / max(xhub_arr), metal_exit_angle_deg=[-55.0], loss_function=FixedPressureLoss(0.05))

    turbine = TurbineSpool(passage, 5.0, inlet, outlet, [stator, rotor], num_streamlines=1, rpm=rpm)
    turbine.adjust_streamlines = False
    return turbine


reference_turbine = build_reference_turbine(rpm)

spec = ComponentSizingSpec(
    inlet_T0=1200.0,           # combustor exit total temperature [K]
    eta_total_guess=0.88,
    exit_static_pressure=101325.0,
    num_stages=1,
    # ngv_choked=True (default): station 4 sized at exactly M=1
)
shaft = ShaftMatch(compressor, spec, reference=reference_turbine, fuel_air_ratio=0.0, bleed_fraction=0.0)

sizing = shaft.size()
print(f"NGV throat Mach: {sizing.station_M[0]:.2f}  (choked = {spec.ngv_choked})")
print(f"NGV throat area: {sizing.station_area[0]:.6f} m^2")
print(f"Required shaft power: {sizing.required_power/1000:.2f} kW")

result = shaft.match(tol_rel=5e-3, max_iter=40)
print(f"\nconverged: {result.converged}  ({result.iterations} iterations)")
print(f"massflow residual: {result.massflow_residual:+.4%}")
print(f"power residual:    {result.power_residual:+.4%}")

## 4. Sweep the compressor to build a map

`sweep_operating_points` mutates a spool in place across a range of massflows, re-solving at each
one and collecting the resulting pressure ratio, efficiency, and choke margin - the standard
compressor characteristic. Because `choke_margin` is wired into every row on every solve (from the
same non-dimensional mass flow function used in step 1), a point that can't physically pass its
target massflow is recorded as `converged=False` with the reason, instead of aborting the whole
sweep.

**We sweep the compressor here, not the turbine.** In pressure-balance mode (fixed blade angles,
fixed boundary pressures), `spool.massflow` is only a solver *seed* for a `TurbineSpool` - the
achieved massflow is an output of the geometry and boundary pressures, not something
`massflow_points` can dictate. That's exactly the wrinkle `ShaftMatch.match()` had to solve with two
knobs (area, exit pressure) instead of one. A compressor's massflow target genuinely drives its
row-to-row pressure balance, so it sweeps meaningfully; see `operating_map.py`'s module docstring
for the full explanation.

Only the massflow/choke axis is guaranteed physically honest away from the design point here - the
efficiency axis reuses the design-point loss coefficient at every swept point, since these blade
rows use `FixedPressureLoss` rather than an incidence-responsive correlation. See the "Sweeping an
operating map" section of `mass_flow_function.md` for what that would take.

In [ ]:
design_mdot = compressor._all_rows()[1].total_massflow_no_coolant

massflow_points = np.linspace(design_mdot * 0.85, design_mdot * 1.5, 12)
points = sweep_operating_points(compressor, massflow_points, [compressor.rpm])

converged = [p for p in points if p.converged]
print(f"{len(converged)}/{len(points)} points converged")
for p in points:
    status = f"PR={p.pressure_ratio:.3f}  choke_margin={p.choke_margin_min:+.3f}" if p.converged else f"FAILED: {p.error[:60]}"
    print(f"  mdot={p.massflow:6.2f} kg/s  {status}")

In [ ]:
import matplotlib.pyplot as plt

converged_pts = [p for p in points if p.converged]
mdot = [p.corrected_massflow for p in converged_pts]
pr = [p.pressure_ratio for p in converged_pts]
margin = [p.choke_margin_min for p in converged_pts]

fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(mdot, pr, c=margin, cmap="viridis", s=60)
ax.plot(mdot, pr, "-", color="gray", alpha=0.4, zorder=0)
ax.set_xlabel(r"corrected massflow, $\dot m \sqrt{T_0}/P_0$")
ax.set_ylabel("pressure ratio")
ax.set_title("Compressor characteristic (single speed line)")
cb = fig.colorbar(sc, ax=ax)
cb.set_label("choke margin (min over rows)")
plt.show()